In [19]:
from typing import List

from langchain_core.runnables import RunnableLambda
from typing_extensions import TypedDict


class Args(TypedDict):
    a: int
    b: List[int]


def f(x: Args) -> str:
    return str(x["a"] * max(x["b"]))


runnable = RunnableLambda(f)
as_tool = runnable.as_tool(
    name="My tool",
    description="Explanation of when to use tool.",
)

In [20]:
print(as_tool.description)

as_tool.args_schema.schema()

Explanation of when to use tool.


{'title': 'My tool',
 'type': 'object',
 'properties': {'a': {'title': 'A', 'type': 'integer'},
  'b': {'title': 'B', 'type': 'array', 'items': {'type': 'integer'}}},
 'required': ['a', 'b']}

In [21]:
as_tool.invoke(
    {
        "a": 3,
        "b": [1, 2],
    }
)

'6'

In [22]:
from typing import Any, Dict


def g(x: Dict[str, Any]) -> str:
    return str(x["a"] * max(x["b"]))


runnable = RunnableLambda(g)
as_tool = runnable.as_tool(
    name="My tool",
    description="Explanation of when to use tool.",
    arg_types={
        "a": int,
        "b": List[int],
    },
)

In [23]:
from langchain_core.pydantic_v1 import BaseModel, Field


class GSchema(BaseModel):
    """Apply a function to an integer and list of integers."""

    a: int = Field(..., description="Integer")
    b: List[int] = Field(..., description="List of ints")


runnable = RunnableLambda(g)
as_tool = runnable.as_tool(GSchema)

In [24]:
def f(x: str) -> str:
    return x + "a"


def g(x: str) -> str:
    return x + "z"


runnable = RunnableLambda(f) | g
as_tool = runnable.as_tool()

In [25]:
as_tool.invoke("b")

'baz'

In [26]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

llm = ChatOpenAI()

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
    ),
]

vectorstore = InMemoryVectorStore.from_documents(
    documents,
    embedding=OpenAIEmbeddings(),
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1},
)

In [27]:
from langgraph.prebuilt import create_react_agent

tools = [
    retriever.as_tool(
        name="pet_info_retriever",
        description="Get information about pets.",
    )
]
agent = create_react_agent(
    llm,
    tools,
)

In [28]:
from langgraph.prebuilt import create_react_agent

tools = [
    retriever.as_tool(
        name="pet_info_retriever",
        description="Get information about pets.",
    )
]
agent = create_react_agent(
    llm,
    tools,
)

In [29]:
for chunk in agent.stream(
    {
        "messages": [
            (
                "human",
                "What are dogs known for?",
            )
        ]
    }
):
    print(chunk)
    print("----")

{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_RkzheURs0dmYWgv4fge7SXGM', 'function': {'arguments': '{"config":{"tags":["dogs"]}}', 'name': 'pet_info_retriever'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 96, 'total_tokens': 116}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-2541dac1-5f3a-4b0b-afc4-2faa089f0613-0', tool_calls=[{'name': 'pet_info_retriever', 'args': {'config': {'tags': ['dogs']}}, 'id': 'call_RkzheURs0dmYWgv4fge7SXGM', 'type': 'tool_call'}], usage_metadata={'input_tokens': 96, 'output_tokens': 20, 'total_tokens': 116})]}}
----
{'tools': {'messages': [ToolMessage(content='Error: TypeError("argument \'text\': \'dict\' object cannot be converted to \'PyString\'")\n Please fix your mistakes.', name='pet_info_retriever', tool_call_id='call_RkzheURs0dmYWgv4fge7SXGM')]}}
----
{'agent': {'messages': 

In [30]:
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

system_prompt = """
You are an assistant for question-answering tasks.
Use the below context to answer the question. If
you don't know the answer, say you don't know.
Use three sentences maximum and keep the answer
concise.

Answer in the style of {answer_style}.

Question: {question}

Context: {context}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            system_prompt,
        )
    ]
)

rag_chain = (
    {
        "context": itemgetter("question") | retriever,
        "question": itemgetter("question"),
        "answer_style": itemgetter("answer_style"),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [31]:
rag_chain.input_schema.schema()

{'title': 'RunnableParallel<context,question,answer_style>Input',
 'type': 'object',
 'properties': {'question': {'title': 'Question'},
  'answer_style': {'title': 'Answer Style'}}}

In [32]:
rag_tool = rag_chain.as_tool(
    name="pet_expert",
    description="Get information about pets.",
)

In [33]:
agent = create_react_agent(llm, [rag_tool])

for chunk in agent.stream(
    {
        "messages": [
            (
                "human",
                "What would a pirate say dogs are known for?",
            )
        ]
    }
):
    print(chunk)
    print("----")

{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_ICfkj0gvu9fXR2OcWggbGA1P', 'function': {'arguments': '{"question":"What are dogs known for according to pirates?","answer_style":"quote"}', 'name': 'pet_expert'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 59, 'total_tokens': 87}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-2386f0a8-13bc-47bc-9ca3-cfecebfea13c-0', tool_calls=[{'name': 'pet_expert', 'args': {'question': 'What are dogs known for according to pirates?', 'answer_style': 'quote'}, 'id': 'call_ICfkj0gvu9fXR2OcWggbGA1P', 'type': 'tool_call'}], usage_metadata={'input_tokens': 59, 'output_tokens': 28, 'total_tokens': 87})]}}
----
{'tools': {'messages': [ToolMessage(content='"Pirates often valued dogs for their loyalty and companionship on long sea voyages."', name='pet_expert', tool_call_id='call_ICfkj0g